# 42. Launch-to-noise spectral ratios across the network

Which frequency bands carry the rocket signal, and which are dominated by background? For every
instrument-corrected channel this notebook compares the launch window with a pre-launch noise window
of the same length:

* **spectral ratio** SNR(f) = PSD(launch) / PSD(noise);
* the **usable band**: frequencies where SNR exceeds 10 (10 dB);
* the **share of rocket energy** (launch minus noise PSD, clipped at zero) in frequency bands, in
  particular below 0.5 Hz, to judge whether a 0.5 Hz high-pass loses rocket signal or only noise.

Gem clipped intervals (notebook 20) are excluded: for a clipped Gem the launch window starts after its
last clipped interval, and the channel is marked.

Outputs (`cfg.OUT_DIR`): `snr_spectra.csv`, `snr_bands.csv`, `snr_band_energy.csv`,
`snr_spectra.png`, `snr_band_energy.png`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from scipy.signal import welch

import artemis2_config as cfg

L = cfg.LAUNCHTIME
OUT = cfg.OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'axes.grid': True, 'grid.color': '#dddddd', 'grid.linewidth': 0.6,
                     'axes.spines.top': False, 'axes.spines.right': False})

SIGNAL_LEN = 180.0                       # s of launch signal, from launch
NOISE = (L - 900, L - 900 + SIGNAL_LEN)  # same length, 15 min before launch
SEG_S = 20.0                             # Welch segment length, s (0.05 Hz resolution)
SNR_MIN = 10.0                           # "usable" threshold on the PSD ratio
BANDS = [(0.02, 0.1), (0.1, 0.5), (0.5, 1), (1, 5), (5, 20), (20, 50), (50, 250)]
stations = cfg.station_table()

## 1. Data, instrument correction, clipped intervals

In [ ]:
raw = cfg.launch_stream_raw()
clips = cfg.stream_clip_intervals(raw)
overflow = {tr.id: cfg.gem_jump_clusters(tr) for tr in raw if tr.stats.channel.startswith('HD')}
# Low pre-filter corners at 0.005/0.01 Hz so the Centaur spectra extend below 0.02 Hz.
st = cfg.instrument_correct(raw, pre_filt_low=(0.005, 0.01))
print(st.__str__(extended=True))

## 2. Spectra

In [ ]:
def welch_psd(x, fs):
    nper = int(2 ** np.round(np.log2(SEG_S * fs)))
    return welch(x - x.mean(), fs, nperseg=nper)

rows, spec = [], {}
for tr in st:
    sid, cha, sta = tr.id, tr.stats.channel, tr.stats.station
    kind = 'seismic' if cha[1] in 'HP' and cha[0] == 'D' else 'infrasound'
    t_start = L
    ivs = clips.get(sid, [])
    over = [c for c in overflow.get(sid, []) if c['kind'] == 'overflow']
    if over:   # start 20 s after the last overflow (residual below 1 Hz decays over ~15 s)
        t_start = max(L, over[-1]['t1'] + 20)
    sig = tr.slice(t_start, t_start + SIGNAL_LEN).data.astype(float)
    noi = tr.slice(*NOISE).data.astype(float)
    if len(sig) < 0.9 * SIGNAL_LEN * tr.stats.sampling_rate or len(noi) < 0.9 * SIGNAL_LEN * tr.stats.sampling_rate:
        print('skip (short window):', sid); continue
    fs = tr.stats.sampling_rate
    f, ps = welch_psd(sig, fs)
    _, pn = welch_psd(noi, fs)
    fmax = 0.45 * fs
    m = (f > 0) & (f <= fmax)
    spec[sid] = dict(f=f[m], sig=ps[m], noise=pn[m], snr=ps[m] / pn[m], kind=kind,
                     dist=stations.distance_km.get(sta, np.nan), clipped=bool(ivs), t_start=t_start - L)
    for fi, a, b, r in zip(f[m], ps[m], pn[m], ps[m] / pn[m]):
        rows.append(dict(seed_id=sid, kind=kind, f_hz=fi, psd_launch=a, psd_noise=b, snr=r))
pd.DataFrame(rows).to_csv(OUT / 'snr_spectra.csv', index=False)
print(len(spec), 'channels')

## 3. Usable band and energy shares

In [ ]:
def band_int(f, p, lo, hi):
    m = (f >= lo) & (f < hi)
    return np.trapezoid(p[m], f[m]) if m.sum() > 1 else np.nan

brows, erows = [], []
for sid, s in spec.items():
    f, snr = s['f'], s['snr']
    fmin = 1.0 if sid.split('.')[-1].startswith('DP') else (0.1 if sid.split('.')[-1].startswith('HD') else 0.0)
    good = (snr >= SNR_MIN) & (f >= fmin)       # SmartSolo/Gem spectra are pre-filtered below fmin
    peak_f = f[f >= fmin][np.argmax(snr[f >= fmin])]
    brows.append(dict(seed_id=sid, kind=s['kind'], distance_km=s['dist'], clipped=s['clipped'],
                      window_start_s=s['t_start'],
                      f_low_snr10=f[good].min() if good.any() else np.nan,
                      f_high_snr10=f[good].max() if good.any() else np.nan,
                      f_peak_snr=peak_f, max_snr=snr.max(),
                      median_snr_0p5_20Hz=np.median(snr[(f >= 0.5) & (f <= 20)])))
    excess = np.clip(s['sig'] - s['noise'], 0, None)
    tot_x, tot_n = band_int(f, excess, 0, np.inf), band_int(f, s['noise'], 0, np.inf)
    row = dict(seed_id=sid, kind=s['kind'], distance_km=s['dist'], clipped=s['clipped'])
    for lo, hi in BANDS:
        row[f'signal_{lo:g}-{hi:g}Hz_%'] = 100 * band_int(f, excess, lo, hi) / tot_x
    row['signal_below_0.5Hz_%'] = 100 * band_int(f, excess, 0, 0.5) / tot_x
    row['noise_below_0.5Hz_%'] = 100 * band_int(f, s['noise'], 0, 0.5) / tot_n
    erows.append(row)
bands = pd.DataFrame(brows).sort_values(['kind', 'distance_km'])
energy = pd.DataFrame(erows).sort_values(['kind', 'distance_km'])
# Unconnected inputs (e.g. DD1/DD2 at B03, B05, B29) and dead sensors carry no launch signal.
dead = set(bands.loc[bands.median_snr_0p5_20Hz < 3, 'seed_id'])
print('dropped (no launch signal, median SNR 0.5-20 Hz < 3):', sorted(dead))
bands, energy = bands[~bands.seed_id.isin(dead)], energy[~energy.seed_id.isin(dead)]
spec = {k: v for k, v in spec.items() if k not in dead}
bands.to_csv(OUT / 'snr_bands.csv', index=False)
energy.to_csv(OUT / 'snr_band_energy.csv', index=False)
with pd.option_context('display.float_format', '{:.2f}'.format, 'display.width', 250, 'display.max_rows', 200):
    display(bands)
    display(energy)

### Summary: is a 0.5 Hz high-pass safe?

In [ ]:
for kind in ('infrasound', 'seismic'):
    # Centaur only (full low band); B05 excluded (Trillium mass off-centre, notebook 20)
    e = energy[(energy.kind == kind) & ~energy.seed_id.str.contains(r'\.DP|\.HD|B05\.00\.DH')]
    print(f"{kind}: rocket energy below 0.5 Hz: median {e['signal_below_0.5Hz_%'].median():.1f}% "
          f"(range {e['signal_below_0.5Hz_%'].min():.1f}-{e['signal_below_0.5Hz_%'].max():.1f}%); "
          f"noise energy below 0.5 Hz: median {e['noise_below_0.5Hz_%'].median():.0f}%")
    b = bands[bands.kind == kind]
    print(f"   SNR>10 band: low edge median {b.f_low_snr10.median():.2f} Hz, high edge median "
          f"{b.f_high_snr10.median():.1f} Hz; SNR peaks at median {b.f_peak_snr.median():.1f} Hz")
print('(<0.5 Hz shares use Centaur channels only: Gems are high-passed at 0.05 Hz and SmartSolos at ~1 Hz.)')
print('\nCaveat: SmartSolo spectra are only meaningful above ~1 Hz (geophone correction pre-filter).')

## 4. Figures

In [ ]:
norm = colors.Normalize(vmin=0.5, vmax=8)
cmap = plt.get_cmap('viridis_r')
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, kind in zip(axes, ('infrasound', 'seismic')):
    for sid, s in sorted(spec.items(), key=lambda kv: -np.nan_to_num(kv[1]['dist'])):
        if s['kind'] != kind or not np.isfinite(s['dist']):
            continue
        f = s['f']
        if sid.split('.')[-1].startswith('DP'):
            keep = f >= 1.0
        else:
            keep = f >= 0.02
        ax.loglog(f[keep], s['snr'][keep], color=cmap(norm(s['dist'])), lw=0.8,
                  ls='--' if s['clipped'] else '-', alpha=0.85)
    ax.axhline(1, color='#666666', lw=0.8)
    ax.axhline(SNR_MIN, color='#666666', lw=0.8, ls=':')
    ax.axvline(0.5, color='#eb6834', lw=1)
    ax.set(xlabel='Frequency (Hz)', title=f'{kind.capitalize()}: PSD(launch) / PSD(noise)', xlim=(0.02, 250))
axes[0].set_ylabel('Spectral ratio')
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=axes, label='Distance from LC39B (km)', shrink=0.85)
fig.suptitle('Dashed: Gem clipped during launch (window starts after clipping). Orange: 0.5 Hz. Dotted: SNR 10',
             x=0.01, ha='left', fontsize=10)
fig.savefig(OUT / 'snr_spectra.png', dpi=200, bbox_inches='tight')

In [ ]:
band_cols = [c for c in energy.columns if c.startswith('signal_') and 'below' not in c]
greys = plt.get_cmap('Blues')(np.linspace(0.25, 1, len(band_cols)))
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True)
for ax, kind in zip(axes, ('infrasound', 'seismic')):
    e = energy[(energy.kind == kind) & energy.distance_km.notna()].sort_values('distance_km', ascending=False)
    left = np.zeros(len(e))
    for c, col in zip(band_cols, greys):
        v = e[c].fillna(0).to_numpy()
        ax.barh(range(len(e)), v, left=left, color=col, edgecolor='white', linewidth=1,
                label=c.replace('signal_', '').replace('_%', ''))
        left += v
    ax.set_yticks(range(len(e)))
    ax.set_yticklabels([f"{s.replace('1R.', '')} {d:.1f} km" for s, d in zip(e.seed_id, e.distance_km)], fontsize=7)
    ax.set(xlabel='Share of rocket-signal energy (%)', title=kind.capitalize(), xlim=(0, 100))
    ax.grid(axis='y', visible=False)
axes[1].legend(frameon=False, fontsize=8, title='Band', loc='lower right')
fig.tight_layout()
fig.savefig(OUT / 'snr_band_energy.png', dpi=200)